# A.X.I.S — Autonomous eXecution & Intelligence System

Run each cell **top to bottom**. After Cell 5 you will get a public URL — open it in any browser.

| Component | Details |
|---|---|
| LLM | Ollama llama3:8b on Colab T4 GPU |
| Memory | ChromaDB persistent vector store |
| Backend | FastAPI + Socket.io |
| Public URL | Cloudflare TryCloudflare (Free, No Auth Token Required) |

In [ ]:
# ============================================================
# CELL 1 — Smart Ollama Installation (Resolves Docker/Colab env)
# ============================================================
import os
import requests
import subprocess
import time
import shutil

OLLAMA_BIN = '/usr/local/bin/ollama'
os.environ['PATH'] = '/usr/local/bin:' + os.environ.get('PATH', '')

def install_ollama():
    if os.path.exists(OLLAMA_BIN):
        print('✅ Ollama binary already exists.')
        return

    # Try official installer script first
    print('🚀 Running official Ollama installation script...')
    # Ignore exit code because it attempts to start systemd at the end, which fails in Colab
    os.system('curl -fsSL https://ollama.com/install.sh | sh')

    if os.path.exists(OLLAMA_BIN):
        print('✅ Ollama successfully installed via script.')
        return

    # Fallback: Manual download and extraction
    print('⚠️ Script installation did not produce binary. Attempting manual download...')
    try:
        # Install zstd package
        print('📦 Installing zstd decompression tool...')
        os.system('apt-get update -qq && apt-get install -y -qq zstd')
        
        # Fetch release info
        res = requests.get('https://api.github.com/repos/ollama/ollama/releases/latest')
        res.raise_for_status()
        release_data = res.json()
        
        # Find the linux-amd64 asset
        download_url = None
        asset_name = None
        for asset in release_data.get('assets', []):
            name = asset.get('name', '')
            if 'linux-amd64' in name and not any(x in name for x in ['rocm', 'mlx', 'cuda']):
                download_url = asset.get('browser_download_url')
                asset_name = name
                break

        if not download_url:
            raise RuntimeError('Could not find linux-amd64 asset in latest release.')

        print(f'📥 Downloading {asset_name}...')
        temp_file = f'/tmp/{asset_name}'
        r = requests.get(download_url, stream=True)
        r.raise_for_status()
        with open(temp_file, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

        # Extract using tar and zstd
        extract_dir = '/tmp/ollama_extracted'
        os.makedirs(extract_dir, exist_ok=True)
        subprocess.run(['tar', '--use-compress-program=zstd', '-xf', temp_file, '-C', extract_dir], check=True)
        
        # Find binary
        found_binary = None
        for root, _, files in os.walk(extract_dir):
            if 'ollama' in files:
                found_binary = os.path.join(root, 'ollama')
                break
        
        if found_binary:
            shutil.copy2(found_binary, OLLAMA_BIN)
            os.chmod(OLLAMA_BIN, 0o755)
            print('✅ Ollama binary successfully installed via manual fallback.')
        else:
            raise RuntimeError('Ollama binary not found in extracted package.')
            
    except Exception as e:
        raise RuntimeError(f'❌ All installation methods failed. Error: {e}')

# Execute
install_ollama()

# Start server
print('Starting Ollama background server...')
subprocess.Popen(
    [OLLAMA_BIN, 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(6)  # Wait for server to bind to port 11434

try:
    r = requests.get('http://localhost:11434')
    print('✅ Ollama is running:', r.text.strip())
except Exception as e:
    print('❌ Ollama not responding:', e)

In [ ]:
# ============================================================
# CELL 2 — Pull the LLM model
# ============================================================
MODEL = 'llama3:8b'

print(f'Pulling {MODEL}...')
result = subprocess.run([OLLAMA_BIN, 'pull', MODEL], capture_output=True, text=True)
print(result.stdout[-500:] if result.stdout else 'Done')

models = subprocess.run([OLLAMA_BIN, 'list'], capture_output=True, text=True)
print('Installed models:\n', models.stdout)

In [ ]:
# ============================================================
# CELL 3 — Install Python packages
# ============================================================
print('Installing Python dependencies...')
os.system('pip install -q fastapi uvicorn python-socketio chromadb requests httpx psutil')
print('Done.')

In [ ]:
# ============================================================
# CELL 4 — Clean clone/pull the AXIS_AI project from GitHub
# ============================================================
REPO_URL = 'https://github.com/aaweshdas/AXIS-AI.git'

if not os.path.exists('/content/AXIS_AI'):
    print('Cloning A.X.I.S project...')
    os.system(f'git clone {REPO_URL} /content/AXIS_AI')
else:
    print('Project already cloned. Force pulling latest updates...')
    # git reset avoids conflicts from the previous local patching scripts
    os.system('git -C /content/AXIS_AI reset --hard HEAD')
    os.system('git -C /content/AXIS_AI pull')

os.chdir('/content/AXIS_AI')
print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
# ============================================================
# CELL 5 — Start FastAPI + Socket.io and Cloudflare Tunnel
# ============================================================
import threading
import sys
import uvicorn
import time
import re

sys.path.insert(0, '/content/AXIS_AI')
from server import socket_app, MODEL

PORT = 8000

# 1. Start FastAPI backend server
def run_server():
    uvicorn.run(socket_app, host='0.0.0.0', port=PORT, log_level='warning')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)

# 2. Download Cloudflare tunnel binary if not already present
CF_BIN = '/usr/local/bin/cloudflared'
if not os.path.exists(CF_BIN):
    print('📥 Installing Cloudflare tunnel...')
    os.system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O ' + CF_BIN)
    os.chmod(CF_BIN, 0o755)

# 3. Open Cloudflare Tunnel and parse the public URL
print('🌐 Creating secure tunnel to public web...')
cf_process = subprocess.Popen(
    [CF_BIN, 'tunnel', '--url', f'http://localhost:{PORT}'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

public_url = None
for _ in range(50):
    line = cf_process.stdout.readline()
    if not line:
        break
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

if not public_url:
    raise RuntimeError('❌ Cloudflare Tunnel failed to start. Please restart the cell.')

print('\n' + '='*55)
print('  A.X.I.S is LIVE:')
print(f'  {public_url}/ui')
print('='*55)
print(f'  Model  : {MODEL}')
print(f'  Health : {public_url}/api/health')
print('='*55 + '\n')

print('Ready! Click the trycloudflare link above to start chatting.')

In [ ]:
# ============================================================
# CELL 6 — Health check
# ============================================================
import requests as req

print('Health checks:\n')

try:
    r = req.get('http://localhost:11434', timeout=3)
    print('Ollama :', r.text.strip())
except Exception:
    print('Ollama : NOT reachable')

try:
    r = req.get(f'http://localhost:{PORT}/api/health', timeout=5)
    d = r.json()
    print(f'FastAPI: online | model={d["model"]} | memories={d["memory_entries"]}')
except Exception as e:
    print(f'FastAPI: {e}')

print('\nTunnel is active. Open the /ui URL from Cell 5 in your browser!')